In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
from matplotlib.ticker import FormatStrFormatter
import io
from PIL import Image
from IPython.display import display, Image as IPImage
from pathlib import Path

import importlib, utils
importlib.reload(utils)
from utils import *

np.random.seed(42)

In [2]:
SetStyle()

# This Notebook loads the preprocessed files and prepare the histograms and datacards required to perform the fits. You can change the overall definitions to test different combinations of data regions, observables, and binning. 

# Event Info input order
* log(pt), eta, phi, log(mass) [0-3]
* dijet mass [4]
* multiplicity, tau1, tau2, tau3, tau4, btag, x(bb), x(qq), x(cc) [5 - 13]
* dijet trigger, muon trigger [14, 15]
* N muons, N electrons [16, 17]
* pt JES Up, mass JES Up, pt JES Down, mass JES Down, pt JER Up, mass JER Up, pt JER Down, mass JER Down [18 - 33]
* Nominal weight, pu Up, pu Down, prefiring Up, prefiring Down, top pt Up, top pt Down, ISR Up, ISR Down, FSR Up, FSR Down, scale weights [34-51]

In [3]:
size = 'l'
pred_type = "NP"
observable = 'dimass'
sort_type = "pt"
year = "2016"
do_sys = True
signal_name = "dihiggs_dijet"
tag = 'hh_signal'
eff = 0.002
data_folder = "/pscratch/sd/v/vmikuni/Omnilearned/"
datacard_path = "/pscratch/sd/v/vmikuni/CMSSW_14_1_0_pre4/NP/"
#Define which regiosn to write to datacards:
# SR: low tau21
# VR1: high tau21
# SR1: low tau21 + subleading jet mass > 100 GeV
# SR2: low tau21 + subleading jet mass < 100 GeV
# SR3: low tau21 + subleading jet mass > 100 GeV + at least one b-tagged jet
# SR4: low tau21 + at least one b-tagged jet
datacard_regions = ['SR1','SR2','VR1'] 
#Binnings for different regions of interest
if observable == 'dimass':
    bins = {
        "SR": np.linspace(1000,1600,15),
        "VR1": np.linspace(1000,1600,20),
        "SR1": np.linspace(1000,1600,16),
        "SR2": np.linspace(1000,1600,8),
        }
else:
    bins = {
        "SR": np.linspace(60,240,20),
        "VR1": np.linspace(60,240,20),
        "SR1": np.linspace(60,240,18),
        "SR2": np.linspace(60,240,7),
        "SR3": np.linspace(60,240,18),
        "SR4": np.linspace(60,235,7),
        
        }
    
mc_names = ['top_dijet','ww_dijet','wz_dijet',"zz_dijet","higgs_dijet",
            'wjets_dijet','zjets_dijet',"topw_dijet", "topz_dijet", "stopw_dijet", "stop_dijet"]   
if signal_name == "dihiggs_dijet":
    mc_names.append("dihiggs_dijet")

    

get_eff_cut = True
look_at_data = False
plot_kde = False
Path("plots").mkdir(parents=True, exist_ok=True)

# First we determine the cut threshold to be applied to the AD score given a fixed efficiency

In [4]:
if get_eff_cut:
    data_plot = EventData.from_npz_folder(name="data",pred_type=pred_type,sort_type="pt",region_type="SR3",
                                          folder=data_folder,pattern=f"outputs_pretrain_{size}_jetht_dijet.npz",
                                          observable=observable)
    cut = np.quantile(np.min(data_plot.prediction[data_plot.base_mask],1), 1.0 - eff)
    print(f"The cut threshold for eff {eff} is: {cut}")

The cut threshold for eff 0.002 is: 0.9017606973648071


# Next, we can take a look at the observable distribution after the selection

In [5]:
if look_at_data:   
    mask_sr, mask_cr1, mask_cr3, mask_cr2 = abcd_pred(data_plot.prediction[:, 0], data_plot.prediction[:, 1], cut, cut, data_plot.mask)
    fig, ax = plt.subplots(figsize=(8, 7))  
    #data_plot.mass[:,0] = np.clip(data_plot.mass[:,0], 0,bin[-1]-1e-5)
    n = ax.hist(data_plot.obs[mask_sr][:,0].flatten(),bins=bins['SR'])[0]

# We can also see how the distribution shape changes for different efficiency thresholds

In [6]:
if look_at_data:
    effs = np.geomspace(0.99, 0.0005, 100)
    frames = []

    for ef in effs:
        sel = np.quantile(np.min(data_plot.prediction[data_plot.base_mask],1), 1.0 - ef)
        mask_sr, mask_cr1, mask_cr3, mask_cr2 = abcd_pred(
            data_plot.prediction[:, 0],
            data_plot.prediction[:, 1],
            sel, sel,
            data_plot.mask
        )

        fig, ax = plt.subplots(figsize=(8, 7)) 

        ax.hist(data_plot.obs[mask_sr][:, 0].flatten(), bins=bins['SR'], color='gray', edgecolor=None,alpha=0.3)
        ax.set_xlabel("Leading Jet Mass [GeV]")
        ax.set_ylabel("Events / bin")
        
        fig.suptitle("Anomalous Region  |  eff = {:.4f}".format(ef))

        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=80, bbox_inches="tight")
        buf.seek(0)
        frames.append(Image.open(buf).copy())
        plt.close(fig)

    gif_buf = io.BytesIO()
    frames[0].save(
        gif_buf,
        format="GIF",
        save_all=True,
        append_images=frames[1:],
        duration=500,
        loop=0,
        disposal=2,
    )
    gif_buf.seek(0)
    with open("plots/efficiency_scan.gif", "wb") as f:
        f.write(gif_buf.read())
    gif_buf.seek(0)  # rewind again so display() still works
    display(IPImage(data=gif_buf.read()))

# We Can also make the KDE plots to verify the correlations between jets and the invariant mass

In [7]:
if plot_kde:
    data_plot = EventData.from_npz_folder(name="data",pred_type=pred_type,sort_type="pt",region_type="SR4",
                                          folder=data_folder,pattern=f"outputs_pretrain_{size}_jetht_dijet.npz",
                                          observable=observable)
    cut = np.quantile(np.min(data_plot.prediction[data_plot.base_mask],1), 1.0 - eff)
    mask_sr, mask_cr1, mask_cr3, mask_cr2 = abcd_pred(
        data_plot.prediction[:, 0],
        data_plot.prediction[:, 1],
        cut, cut,
        data_plot.mask
    )

    from scipy.stats import gaussian_kde
    def make_kde(x, y, xrange, yrange, bw=0.20, ngrid=50, weights=None):
        kde = gaussian_kde(np.vstack([x, y]), bw_method=bw, weights=weights)
        xx, yy = np.meshgrid(
            np.linspace(xrange[0], xrange[1], ngrid),
            np.linspace(yrange[0], yrange[1], ngrid)
        )
        zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
        zz /= len(x)
        return xx, yy, zz

    def plot_kde(data_x, data_y, xrange, yrange, xlabel, ylabel, outname):
        xx, yy, zz = make_kde(data_x, data_y, xrange, yrange)

        fig, ax = plt.subplots(figsize=(8, 8))

        # Save contourf handle
        cf = ax.contourf(xx, yy, zz, levels=10, cmap="magma")
        ax.contour(xx, yy, zz, levels=10, colors="white", linewidths=0.4, alpha=0.3)

        # Add colorbar (z-axis)
        cbar = fig.colorbar(cf, ax=ax)
        cbar.set_label("Density", fontsize=16)

        ax.set_xlim(xrange[0], xrange[1])
        ax.set_ylim(yrange[0], yrange[1])
        ax.set_xlabel(xlabel, fontsize=16)
        ax.set_ylabel(ylabel, fontsize=16)
        ax.grid(True, linestyle="--", alpha=0.4)

        plt.tight_layout()

        # Save BEFORE show
        plt.savefig(outname, format="pdf")
        plt.show()

    # 1) obs[:,0] vs obs[:,1]
    data_x = data_plot.mass[:, 0][mask_sr]
    data_y = data_plot.mass[:, 1][mask_sr]

    plot_kde(
        data_x,
        data_y,
        xrange=[60, 240],
        yrange=[60, 240],
        xlabel="Leading Jet Soft Drop Mass [GeV]",
        ylabel="Subleading Jet Soft Drop Mass [GeV]",
        outname="plots/kde_pt_pt.pdf",
    )

    # 2) obs[:,0] vs dimass[:,0]
    mask = mask_sr * (data_plot.mass[:, 1] > 100)

    data_x = data_plot.mass[:, 0][mask]
    data_y = data_plot.dimass[:, 0][mask]

    plot_kde(
        data_x,
        data_y,
        xrange=[60, 240],
        yrange=[1000, 1500],
        xlabel="Leading Jet Soft Drop Mass [GeV]",
        ylabel=r"Dijet Invariant Mass [GeV]",
        outname="plots/kde_pt_mhh.pdf",
    )

# Now we run the main datacard creation loop. 

In [8]:
for region_type in datacard_regions:
    data = EventData.from_npz_folder(name="data",pred_type=pred_type,sort_type=sort_type,region_type=region_type,
                                     folder=data_folder,pattern=f"outputs_pretrain_{size}_jetht_dijet.npz",
                                     observable=observable)
    mcs = {}
    mcs_sys = {}
    for name in mc_names:
        mcs[name] = EventData.from_npz_folder(name=name,pred_type=pred_type,sort_type=sort_type,region_type=region_type,
                                              folder=data_folder,pattern=f"outputs_pretrain_{size}_{name}.npz",
                                              observable=observable)
        if do_sys:
            for jec in ['jes_up','jes_down','jer_up','jer_down', 'jms_up', 'jms_down', 'jmr_up', 'jmr_down']:
                mcs_sys[f"{name}_{jec}"] = EventData.from_npz_folder(name=name,pred_type=pred_type,sort_type=sort_type,region_type=region_type,
                                              folder=data_folder,pattern=f"outputs_pretrain_{size}_{name}.npz",sys=jec,
                                              observable=observable)
                  
    print("Pass dijet check in data: ", np.all(data.dimass[:,0]==data.dimass[:,1]))
    for mc in mcs:
        print(f"Pass dijet check in {mc}: ", np.all(mcs[mc].dimass[:,0]==mcs[mc].dimass[:,1]))
        
    root_file = f"{datacard_path}/combine_histograms_{pred_type}_{region_type}_{size}_{observable}_{tag}.root"

    save_combine_histograms(
        output_path=root_file,
        bin_edges=bins[region_type],
        data=data,
        mcs=mcs,
        cut=cut,
        year=year,
        tag=region_type,
        mcs_sys=mcs_sys,
        do_sys=do_sys,
        pred_type=pred_type,
        observable=observable,
        size=size,
    )
    
    (abcd_vals, abcd_sw2,
     cr1_vals,  cr1_sw2,
     cr2_vals,  cr2_sw2,
     cr3_vals,  cr3_sw2) = get_abcd_prediction_per_region(data, mcs, cut, bins[region_type])
    
    data_per_cr = {
        "CR1": (cr1_vals, cr1_sw2),
        "CR2": (cr2_vals, cr2_sw2),
        "CR3": (cr3_vals, cr3_sw2),
    }
    


    write_datacard(
        output_path=f"{datacard_path}/datacard_{pred_type}_{region_type}_{size}_{observable}_{tag}.txt",
        root_file=root_file,
        bin_edges=bins[region_type],
        mc_names=mc_names,
        abcd_vals=abcd_vals,
        data_per_cr=data_per_cr,
        year=year,
        tag=region_type,
        do_sys=do_sys,
        qcd_sr_bin_unc=None,
    )
        

Pass dijet check in data:  True
Pass dijet check in top_dijet:  True
Pass dijet check in ww_dijet:  True
Pass dijet check in wz_dijet:  True
Pass dijet check in zz_dijet:  True
Pass dijet check in higgs_dijet:  True
Pass dijet check in wjets_dijet:  True
Pass dijet check in zjets_dijet:  True
Pass dijet check in topw_dijet:  True
Pass dijet check in topz_dijet:  True
Pass dijet check in stopw_dijet:  True
Pass dijet check in stop_dijet:  True
Pass dijet check in dihiggs_dijet:  True
[!] Skipping scale_ww_dijet in SR: envelope is trivial
[!] Skipping scale_ww_dijet in CR1: envelope is trivial
[!] Skipping scale_ww_dijet in CR2: envelope is trivial
[!] Skipping scale_ww_dijet in CR3: envelope is trivial
[!] Skipping scale_wz_dijet in SR: envelope is trivial
[!] Skipping scale_wz_dijet in CR1: envelope is trivial
[!] Skipping scale_wz_dijet in CR2: envelope is trivial
[!] Skipping scale_wz_dijet in CR3: envelope is trivial
[!] Skipping scale_zz_dijet in SR: envelope is trivial
[!] Skippin

# Combine individual datacards before the fit

In [9]:
input_datacards = {}
root_files = {}
for region in datacard_regions:
    input_datacards[region] = f"{datacard_path}/datacard_{pred_type}_{region}_{size}_{observable}_{tag}.txt"
    root_files[region] = f"{datacard_path}/combine_histograms_{pred_type}_{region}_{size}_{observable}_{tag}.root"
 
abcd_vals_dict       = {}
data_per_cr_combined = {}

for r, root_file in root_files.items():
    with uproot.open(root_file) as f:
        qcd_name = f"QCD_{r}"
        n_bins   = len(bins[r]) - 1
        cr1_vals = np.zeros(n_bins)
        cr2_vals = np.zeros(n_bins)
        cr3_vals = np.zeros(n_bins)
        cr1_sw2  = np.zeros(n_bins)
        cr2_sw2  = np.zeros(n_bins)
        cr3_sw2  = np.zeros(n_bins)

        for i in range(n_bins):
            b = i + 1
            for region, vals_arr, sw2_arr in [
                ("CR1", cr1_vals, cr1_sw2),
                ("CR2", cr2_vals, cr2_sw2),
                ("CR3", cr3_vals, cr3_sw2),
            ]:
                h           = f[f"{region}/{qcd_name}_bin_{b}"]
                vals_arr[i] = h.values()[i]
                sw2_arr[i]  = h.variances()[i]

        h_sr     = f[f"SR/{qcd_name}"]
        abcd_v   = h_sr.values()
        abcd_sw2 = h_sr.variances()

    abcd_vals_dict[tag]       = abcd_v
    data_per_cr_combined[tag] = {
        "CR1": (cr1_vals, cr1_sw2),
        "CR2": (cr2_vals, cr2_sw2),
        "CR3": (cr3_vals, cr3_sw2),
    }



write_combined_datacard_from_existing(
    output_path=f"{datacard_path}/datacard_{pred_type}_combined_{size}_{observable}_{tag}.txt",
    input_datacards=input_datacards,
    root_files=root_files,
    bin_edges=bins,
    mc_names=mc_names,
    data_per_cr=data_per_cr_combined,
    abcd_vals_dict=abcd_vals_dict,
    year=year,
    do_sys=do_sys,
    qcd_sr_bin_unc=None,
)
    


[✓] Combined datacard written to: /pscratch/sd/v/vmikuni/CMSSW_14_1_0_pre4/NP//datacard_NP_combined_l_dimass_hh_signal.txt
    Channels        : 12 (3 tags × 4 regions)
    Free-floating   : ['top_dijet']
    Uncorrelated QCD: ['QCD_SR1 (15 bins)', 'QCD_SR2 (7 bins)', 'QCD_VR1 (19 bins)']


# You can now run the fit using the Combine package! Follow the installation instructions from the [official page](https://cms-analysis.github.io/HiggsAnalysis-CombinedLimit/latest/)

# Run the following script inside the cmsenv:
```bash
chmod u+x run_fit.sh
./run_fit.sh --type {pred_type} --size {size} --obs {observable} --tag {tag}
```